### ライブラリの準備

###モジュールのインポートとGoogleドライブのマウント

In [ ]:
import os
import glob
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import datetime
#from tqdm import tqdm
from tqdm.notebook import tqdm
import pickle
import random
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from PIL import Image
import skimage.transform
from collections import deque
from typing import Sequence, Dict, Tuple, Union

import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence
from torchvision import models
import torchvision.transforms as T
import torchvision.datasets as dataset
from torchvision.transforms import v2

from timm.scheduler import CosineLRScheduler
from transformers import  get_linear_schedule_with_warmup

#from transformers import AutoImageProcessor, AutoModel, AutoProcessor, CLIPVisionModel
from transformers import BertTokenizer, BertModel, CLIPVisionModel, BertForPreTraining

import sys

import util
import levenshtein
from nltk import bleu_score
import ssl
from torch.amp import autocast, GradScaler

In [ ]:
class PositionalEmbedding(nn.Module):
    '''
    位置埋め込み （Positional embedding）
    dim_embedding: 埋込み次元
    max_len      : 入力の最大系列長
    '''
    def __init__(self, dim_embedding: int, max_len: int=2048):
        super().__init__()

        self.pos_emb = nn.Embedding(max_len, dim_embedding)

    '''
    位置エンコーディングの順伝播
    x: 位置エンコーディングを埋め込む対象のテンソル,
       [バッチサイズ, 系列長, 埋め込み次元]
    '''
    def forward(self, x: torch.Tensor):
        seq = x.shape[1]
        positions = torch.arange(start=0, end=seq, step=1, device=x.device).to(torch.long)
        positions = self.pos_emb(positions)[:seq,:]
        
        return positions

### CaptioningTransformer

In [ ]:
class CaptioningTransformer(nn.Module):
    '''
    CaptioningTransformerのコンストラクタ
    dim_embedding  : 埋め込み次元
    dim_feedforward: FNNの中間特徴次元
    num_heads      : マルチヘッドアテンションのヘッド数
    num_layers     : Transformerデコーダ層の数
    vocab_size     : 辞書の次元
    null_index     : NULLのID
    dropout        : ドロップアウト確率
    '''
    def __init__(self, img_size: int, length_max: int, dim_embedding: int,
                  vocab_size: int, tokenizer, dropout: float=0.1, model_id: str=''):
        super().__init__()

        self.mask_token_id = tokenizer.mask_token_id
        self.pad_token_id = tokenizer.pad_token_id
        self.max_idx_en = len( tokenizer )

        #CLIP
        clip_model_id = "openai/clip-vit-large-patch14-336"
        self.clip_model = CLIPVisionModel.from_pretrained(clip_model_id, output_hidden_states = True)
        images = torch.randn( ( 1, 3, img_size, img_size ) )
        memory = self.clip_model( images )
        memory = memory.last_hidden_state
        img_length = memory.size(1)
        clip_dim = memory.size(2)
        self.ln_memory = nn.LayerNorm( dim_embedding )

        self.emb = nn.Embedding( vocab_size, dim_embedding, padding_idx=tokenizer.pad_token_id )
        self.pos_emb = PositionalEmbedding( dim_embedding )

        self.dropout = nn.Dropout( dropout )

        self.dc_linear = nn.Linear( clip_dim * 3, dim_embedding )
        #self.dc_ln = nn.LayerNorm( dim_embedding )

        # Down Sampling
        #img_length = 577
        #length_max = 84
        stride = img_length // length_max
        self.conv1 = nn.Conv1d( dim_embedding, dim_embedding, 1, stride )
        print( "img_length:", img_length )
        print( "text_length_max:", length_max )
        print( "stride:", stride )
        seq_len = self.conv1( memory.transpose(1,2) ).size( 2 )
        
        self.bert = BertModel.from_pretrained( model_id )

        ## 単語出力分布計算
        self.ln_outputs = nn.LayerNorm( dim_embedding )
        self.linear = nn.Linear(dim_embedding, vocab_size)

        self.ln_length = nn.LayerNorm( dim_embedding )
        self.conv_length = nn.Conv1d( seq_len, 1, 1 )
        self.embed_lengths = nn.Embedding(1024, dim_embedding)
        nn.init.normal_(self.embed_lengths.weight, mean=0, std=0.02)
        
        self.dim_embedding = dim_embedding

    ''' CaptioningTransformerの順伝播処理
    features: 画像特徴量 [バッチサイズ, 埋め込み次元]
    captions: 正解キャプション [バッチサイズ, 系列長]
    '''
    def forward(self, images: torch.Tensor, captions: torch.Tensor, caption_lengths: torch.Tensor ):

        self.device = images.device

        masked_captions, mask = self.masking( captions, caption_lengths )
        
        memory = self.clip_model( images )
        memory = self.dense_connector( memory )
        memory = self.dropout( memory )
        memory = self.ln_memory( memory )

        memory = self.conv1( memory.transpose(1,2) ).transpose(1,2)
        
        emb_caption = self.emb( masked_captions ) * math.sqrt(self.dim_embedding)
        emb_caption += self.pos_emb( emb_caption )

        bert_in = torch.cat( [memory, emb_caption], dim = 1 )
        bert_in_padding_masks = (~(torch.eq( masked_captions, self.pad_token_id ))).float()
        bert_in_padding_masks = torch.cat( [torch.ones( memory.shape[:2], device=self.device ), bert_in_padding_masks], dim = 1 )
        
        outputs = self.bert( inputs_embeds = bert_in, attention_mask = bert_in_padding_masks ).last_hidden_state
        outputs = outputs[:,memory.size(1):,:]
        outputs = self.ln_outputs( outputs )
        logits = self.linear( outputs )

        predicted_lengths = self.lengths_predictor( memory )
        
        return logits, mask, predicted_lengths

    def dense_connector(self, memory ):
        tmp1 = torch.tensor([], device = self.device )
        tmp2 = torch.tensor([], device = self.device )
        tmp_full = len( memory.hidden_states )
        tmp_half = tmp_full // 2
        for i in range( 0, tmp_half ):
            tmp1 = torch.cat( [tmp1, memory.hidden_states[i][None]], dim = 0 )
        tmp1 = torch.sum(tmp1, dim=0) / tmp_half
        for i in range( tmp_half, tmp_full ):
            tmp2 = torch.cat( [tmp2, memory.hidden_states[i][None]], dim = 0 )
        tmp2 = torch.sum(tmp2, dim=0 ) / ( tmp_full - tmp_half )
        tmp3 = torch.cat([tmp1, tmp2], dim=-1)
        tmp3 = torch.cat( [ memory.last_hidden_state, tmp3], dim = -1 )
        #tmp3 = sel.dc_ln( tmp3 )
        tmp3 = self.dc_linear( tmp3 )
        return tmp3

    def masking(self, input_x: torch.Tensor, lengths: torch.Tensor) -> tuple[torch.Tensor]:

        output = input_x.clone()

        masks = torch.zeros_like( output, device=output.device, dtype=torch.bool )       
        
        #sum_num_mask = 0
        #sum_num_arbi = 0
        #sum_num_nochange = 0
        for n in range( output.size(0) ):
            all_prob = torch.rand( (1) )
            if all_prob > 0.99:
                num_mask = lengths[n]
                num_arbi = 0
                num_nochange = 0
            else:
                mask_prob0 = torch.rand( (1) )
                mask_prob = all_prob * mask_prob0
                resi_prob = all_prob * ( 1.0 - mask_prob0 )
                arbi_prob = all_prob * ( resi_prob * 0.5 )
                nochange_prob = all_prob * ( resi_prob * 0.5 )
                num_mask = math.floor( lengths[n].item() * mask_prob )
                num_arbi = math.floor( lengths[n].item() * arbi_prob )
                num_nochange = math.floor( lengths[n].item() * nochange_prob )

            #sum_num_mask += num_mask
            #sum_num_arbi += num_arbi
            #sum_num_nochange += num_nochange
            
            mask_mask = list( random.sample( list(range( 0, lengths[n])),  num_mask ))
            output[n,mask_mask] = self.mask_token_id
            not_mask_mask = [ n for n in range( lengths[n] ) if n not in mask_mask ]
            mask_arbi = random.sample( not_mask_mask, num_arbi )
            for i in range( lengths[n] ):
                if i in mask_arbi:
                    output[n,i] = torch.randint( 0, self.max_idx_en, size=(1,))
            not_mask_arbi = [ n for n in not_mask_mask if n not in mask_arbi ]
            mask_nochange = random.sample( not_mask_arbi, num_nochange )
            not_mask_nochange = [ n for n in not_mask_arbi if n not in mask_nochange ]
            mask = [ False if n in not_mask_nochange else True for n in range(lengths[n]) ]
            masks[n,:lengths[n]] = torch.tensor( mask )

        #print( "sum_num_mask:", sum_num_mask )
        #print( "calculate num mask:", torch.sum( torch.eq( output, self.mask_token_id ).int() ) )
        #print( "sum_num_mask + sum_num_arbi :", sum_num_mask + sum_num_arbi )
        #print( "num not equal:", torch.sum( torch.ne( input_x, output ).int() ) )
        #print( "sum_num_mask + sum_num_arbi + sum_nochange:", sum_num_mask + sum_num_arbi + sum_num_nochange )
        #print( "num of mask True:", torch.sum( torch.eq( masks, True ) ) )
        
        return output, masks
        
    def lengths_predictor(self, memory):
        
        x = self.ln_length(memory)
        x = self.conv_length( x )
        #print( "size of x[:,0,:]:",x[:,0,:].size())
        #print( "size of self.pos_emb.pos_emb.weight.tranpose(0,1):",self.pos_emb.pos_emb.weight.transpose(0,1).size())
        predicted_lengths_logits = torch.matmul( x[:,0,:], self.embed_lengths.weight.transpose(0,1)).float()
        predicted_lengths_logits [:,0] += float('-inf')
        predicted_lengths = F.log_softmax( predicted_lengths_logits, dim = -1 )
        #predicted_lengths は複数の候補が確率とともに
        
        return predicted_lengths

    def my_decode(self, token_list, tokenizer ):

        def my_index( l, x ):
            if x in l:
                return l.index(x)
            else:
                return -1
        if my_index( token_list, tokenizer.sep_token_id ) != -1:
            token_list = token_list[:my_index( token_list, tokenizer.sep_token_id )]
        else:
            token_list = token_list
            
        text = tokenizer.decode( token_list, skip_special_tokens = True )
        
        return text

In [ ]:
class MyDataset(Dataset):
    def __init__(self, file_path: str, img_directory: str, transforms, tokenizer, length_max = None ) -> None:
        super().__init__()
        self.img_directory = img_directory
        self.transforms = transforms
        # TODO: fix to original data
        #画像の前処理
        self.img_file = []
        self.tokens = []
        self.lengths = []
        if length_max == None:
            self.length_max = 0
        else:
            self.length_max = length_max
        length_sum = 0
        with open( file_path, "r" ) as f:
            for i, line in enumerate( f ):
                if i % 100000 == 0:
                    print( "i:", i )
                self.img_file.append(line.split("\t" )[0])
                caption = line.split("\t")[1].replace( "\r\n", "" ).replace( "\n", "").replace( "\r", "" )
                id_tokens = tokenizer.encode( caption )
                length_sum += len( id_tokens )
                if length_max == None:
                    if self.length_max < len( id_tokens ):
                        self.length_max = len( id_tokens )
                    id_tokens = torch.tensor( id_tokens  )
                    self.lengths.append( len( id_tokens ) )
                else:
                    id_tokens = torch.tensor( id_tokens )[:length_max]
                    self.lengths.append( len( id_tokens ) )
                
                self.tokens.append( id_tokens )

                #line = f.readline()
        print("avg len:", length_sum / len( self.tokens ) )    
    
    # ここで取り出すデータを指定している
    def __getitem__(
        self,
        index: int
    ):
        tokens = self.tokens[index]
        img_file = self.img_file[index] + ".jpg"
        img_path = os.path.join( self.img_directory, img_file ) #index番目の画像のパスを取得
        img = Image.open(img_path) #PIL形式で画像を読み込み
        if img.mode != 'RGB':
            img = img.convert("RGB")
        img = self.transforms(img)
        lengths = self.lengths[index]
        
        return img, tokens, lengths

    # この method がないと DataLoader を呼び出す際にエラーを吐かれる
    def __len__(self) -> int:
        return len(self.tokens)

    def length_max(self):
        return self.length_max

In [ ]:
def collate_func(batch: Sequence[Tuple[Union[torch.Tensor, str]]], pad_index ):
    imgs, tokens, lengths = zip(*batch)

    lengths = torch.tensor( lengths )
    
    max_length = torch.max( lengths )
    
    targets = []
    for target in tokens:
        pad_len = max_length - len( target ) 
        input2= F.pad( target, (0, pad_len), mode='constant', value = pad_index)
        targets.append( input2 )
    
    imgs = torch.stack( imgs, dim = 0 )
    targets = torch.stack( targets, dim = 0 )
    
    return imgs, targets, lengths

In [ ]:
# 画像のtransformsを定義
transforms = v2.Compose([
    v2.Resize((336, 336)),
    #v2.AutoAugment(),
    #v2.ToTensor(),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    ## Coco データセット 2017 train の平均と標準偏差
    #v2.Normalize((0.456,0.427,0.401),(0.224,0.219,0.231) )
    ## ImageNetデータセットの平均と標準偏差
    #v2.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
    # clip の preprocessor_config.json の平均と標準偏差
    v2.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711))
])

model_id = "google-bert/bert-large-uncased"
tokenizer = BertTokenizer.from_pretrained(model_id)

# v7 データセット
train_dataset = MyDataset( file_path="../CLIP_LLM_AR/dataset.txt",
                           img_directory = "/mnt/ssd2/v7/img",
                           #img_directory = "smb://192.168.1.2/img/v7/",
                           transforms=transforms, tokenizer = tokenizer, length_max = 84 )

# Subset samplerの生成
test_set, val_set, train_set = util.generate_subset_test_val_train(
    train_dataset, 0.1, 0.1 )
    
# 学習時にランダムにサンプルするためのサンプラー
train_sampler = SubsetRandomSampler(train_set)

# DataLoaderを生成
collate_func_lambda = lambda x: collate_func(x, tokenizer.pad_token_id )

test_loader = torch.utils.data.DataLoader(
                    train_dataset,
                    #batch_size=config.batch_size,
                    batch_size=1,
                    num_workers=0,
                    sampler=test_set,
                    collate_fn=collate_func_lambda)


###学習におけるハイパーパラメータやオプションの設定

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#device = torch.device("cpu")
## 辞書（単語→単語ID）の読み込み
#with open('../PreTrain_Decoder/translateDatasetNTT_blank4_pad0/word_to_id2.pkl', 'rb') as f:
#    word_to_id = pickle.load(f)
#max_idx_en = len( word_to_id )
#word_to_id['<mask>'] = max_idx_en
#mask_value = word_to_id['<mask>']
#start_idx = word_to_id['<start>']
#bert_model_path = 'models--google-bert--bert-large-uncased/snapshots/6da4b6a26a1877e173fca3225479512db81a5e5b'
#tokenizer = BertTokenizer.from_pretrained(pretrained_model_name_or_path = bert_model_path )
model_id = "google-bert/bert-large-uncased"
tokenizer = BertTokenizer.from_pretrained(model_id)
model = CaptioningTransformer(img_size = 336, length_max = 84, dim_embedding=1024, vocab_size=len(tokenizer),
                 tokenizer=tokenizer, dropout=0.1, model_id =model_id).to(device)

PATH = "model/model_bert_mask_curr.pth"
if os.path.isfile(PATH):
    checkpoint = torch.load(PATH, map_location=torch.device('cpu'))
    model.load_state_dict(checkpoint['model_state_dict'])
    print( "load parameters." )

#images = torch.randint( 0, 255, size = (10,3,256,256) )
images = torch.randn( ( 1, 3, 336,336 ), device = device )
captions = torch.randint( 0, len(tokenizer), size= (1, 50 ), device= device )
caption_lengths = torch.tensor( [50], device=device, dtype=torch.int )
outputs, masks, lengths = model( images, captions, caption_lengths )

print( outputs.size() )
print( masks.size() )
print( lengths.size() )

### 推論関数の定義

In [ ]:
def predict_length_beam(gold_target_len, predicted_lengths, length_beam_size):
    if gold_target_len is not None:
        beam_starts = gold_target_len - (length_beam_size - 1) // 2
        beam_ends = gold_target_len + length_beam_size // 2 + 1
        beam = torch.stack([torch.arange(beam_starts[batch], beam_ends[batch], device=beam_starts.device) for batch in range(gold_target_len.size(0))], dim=0)
    else:
        beam = predicted_lengths.topk(length_beam_size, dim=1)[1]
    beam[beam < 2] = 2
    return beam

# 推論モジュール
@torch.no_grad()
def inference( images, tokenizer ):
    # batch_size = 1 でお願いします。

    length_beam_size = 1
    
    model.eval()
    device = images.device

    memory = model.clip_model( images )
    memory = model.dense_connector( memory )
    memory = model.dropout( memory )
    memory = model.ln_memory( memory )
    memory = model.conv1( memory.transpose(1,2) ).transpose(1,2)

    predicted_lengths = model.lengths_predictor( memory )
    memory = memory.repeat( length_beam_size, 1, 1 )
    beam = predict_length_beam( None, predicted_lengths, length_beam_size )
    beam2 = beam.view( -1 )

    masked_captions = torch.ones( (beam2.size(0), torch.max( beam2 ) ), dtype=torch.long, device=device ) * tokenizer.pad_token_id
    
    for n, length in enumerate( beam2 ):
        if int( length ) >= 3:
            masked_captions[n,1:beam2[n]] = tokenizer.mask_token_id
            masked_captions[n,0] = tokenizer.cls_token_id
            masked_captions[n,beam2[n]-1] = tokenizer.sep_token_id
            masked_captions[n,beam2[n]:] = tokenizer.pad_token_id
        else:
            masked_captions[n * length_beam_size * m, 0:1] = tokenizer.mask_token_ie #<end>
    
    emb_caption = model.emb( masked_captions ) * math.sqrt(model.dim_embedding)
    emb_caption += model.pos_emb( emb_caption )
    bert_in = torch.cat( [memory, emb_caption], dim = 1 )
    bert_in_padding_masks = (~(torch.eq( masked_captions, model.pad_token_id ))).float()
    bert_in_padding_masks = torch.cat( [torch.ones( memory.shape[:2], device = device ), bert_in_padding_masks], dim = 1 )

    iter_max = 10
    for i in range( iter_max ):
        outputs = model.bert( inputs_embeds = bert_in, attention_mask = bert_in_padding_masks ).last_hidden_state
        outputs = outputs[:,memory.size(1):,:]
        outputs = model.ln_outputs( outputs )
        logits = model.linear( outputs )
        probabilities = torch.nn.functional.softmax( logits, dim = 2 )
        captions = torch.argmax( logits, dim = 2 )

        if i < iter_max - 1:
            masked_captions = []
            for n in range( outputs.size(0) ):
                max_prob = torch.max( probabilities[n,:,:], dim = 1 ).values
                sorted_max_prob = torch.sort( max_prob, dim = 0 ).values
                masked_caption = captions[n]
                num_mask = torch.sum( torch.eq( masked_caption, tokenizer.mask_token_id ).int() )
                kosuu_mask = math.floor(( iter_max - i - 1 ) * torch.max( predicted_lengths) / iter_max ) 
                if kosuu_mask  - 1 < 0:
                    kosuu_mask = 1
                if num_mask > kosuu_mask:
                    kosuu_mask = num_mask
                thresh = sorted_max_prob[ kosuu_mask - 1 ]
                t_indices = max_prob < thresh
                masked_caption[t_indices] = tokenizer.mask_token_id
                masked_caption[0] = tokenizer.cls_token_id
                masked_caption[beam2[n]-1] = tokenizer.sep_token_id
                masked_caption[beam2[n]:] = tokenizer.pad_token_id
                masked_captions.append( masked_caption )

            masked_captions = torch.stack( masked_captions, dim = 0 )
            emb_captions = model.emb( masked_captions ) * math.sqrt(model.dim_embedding)
            bert_in = torch.cat( [memory, emb_caption], dim = 1 )
            bert_in_padding_masks = (~(torch.eq( masked_captions, model.pad_token_id ))).float()
            bert_in_padding_masks = torch.cat( [torch.ones( memory.shape[:2], device= device ), bert_in_padding_masks], dim = 1 )
        
    return logits

# 推論モジュール
@torch.no_grad()
def inference2( images, length_beam_size, tokenizer ):
    # batch_size = 1 でお願いします。
    
    model.eval()
    device = images.device
    memory = model.clip_model( images )
    memory = model.dense_connector( memory )
    memory = model.dropout( memory )
    memory = model.ln_memory( memory )
    memory = model.conv1( memory.transpose(1,2) ).transpose(1,2)
    
    predicted_lengths = model.lengths_predictor( memory )
    memory = memory[:,None].expand( -1, length_beam_size, -1, -1 )
    memory = memory.view( memory.size(0) * memory.size(1), memory.size(2), memory.size(3) )
    beam = predict_length_beam( None, predicted_lengths, length_beam_size )
    beam2 = beam.view( beam.size(0) * beam.size(1) )

    masked_captions = torch.ones( (beam2.size(0), torch.max( beam2 ) ), dtype=torch.long, device=device ) * tokenizer.pad_token_id

    for n, length in enumerate( beam2 ):
        if int( length ) >= 3:
            masked_captions[n,1:beam2[n]] = tokenizer.mask_token_id
            masked_captions[n,0] = tokenizer.cls_token_id
            masked_captions[n,beam2[n]-1] = tokenizer.sep_token_id
            masked_captions[n,beam2[n]:] = tokenizer.pad_token_id
        else:
            masked_captions[n * length_beam_size * m, 0:1] = tokenizer.mask_token_ie #<end>

    
    emb_caption = model.emb( masked_captions ) * math.sqrt(model.dim_embedding)
    emb_caption += model.pos_emb( emb_caption )
    bert_in = torch.cat( [memory, emb_caption], dim = 1 )
    bert_in_padding_masks = (~(torch.eq( masked_captions, model.pad_token_id ))).float()
    bert_in_padding_masks = torch.cat( [torch.ones( memory.shape[:2], device = device ), bert_in_padding_masks], dim = 1 )

    iter_max = 10
    for i in range( iter_max ):
        outputs = model.bert( inputs_embeds = bert_in, attention_mask = bert_in_padding_masks ).last_hidden_state
        outputs = outputs[:,memory.size(1):,:]
        outputs = model.ln_outputs( outputs )
        logits = model.linear( outputs )
        probabilities = torch.nn.functional.softmax( logits, dim = 2 )
        captions = torch.argmax( logits, dim = 2 )
        max_probs = torch.max( probabilities[:,:,:], dim = 2 ).values

        if i < iter_max - 1:
            masked_captions = []
            for n in range( outputs.size(0) ):
                max_prob = torch.max( probabilities[n,:,:], dim = 1 ).values
                sorted_max_prob = torch.sort( max_prob, dim = 0 ).values
                masked_caption = captions[n]
                num_mask = torch.sum( torch.eq( masked_caption, tokenizer.mask_token_id ).int() )
                kosuu_mask = math.floor(( iter_max - i - 1 ) * torch.max( predicted_lengths) / iter_max ) 
                if kosuu_mask  - 1 < 0:
                    kosuu_mask = 1
                if num_mask > kosuu_mask:
                    kosuu_mask = num_mask
                thresh = sorted_max_prob[ kosuu_mask - 1 ]
                t_indices = max_prob < thresh
                masked_caption[t_indices] = tokenizer.mask_token_id
                masked_caption[0] = tokenizer.cls_token_id
                masked_caption[beam2[n]-1] = tokenizer.sep_token_id
                masked_caption[beam2[n]:] = tokenizer.pad_token_id
                masked_captions.append( masked_caption )

            masked_captions = torch.stack( masked_captions, dim = 0 )
            emb_captions = model.emb( masked_captions ) * math.sqrt(model.dim_embedding)
            bert_in = torch.cat( [memory, emb_caption], dim = 1 )
            bert_in_padding_masks = (~(torch.eq( masked_captions, model.pad_token_id ))).float()
            bert_in_padding_masks = torch.cat( [torch.ones( memory.shape[:2], device= device ), bert_in_padding_masks], dim = 1 )

    mean_prob = torch.zeros( beam2.size(0) )
    for n, length in enumerate( beam2 ):
        mean_prob[n] = torch.mean( max_probs[n,:length], dim = -1 )
    
    return captions, mean_prob

In [ ]:
def duplicate_encoder_out(encoder_out, encoder_padding_mask, decoder_padding_mask, causal_mask, bsz, beam_size):
    encoder_out = encoder_out.unsqueeze(1).repeat(1, beam_size, 1, 1 ).view( bsz * beam_size, encoder_out.size(1), encoder_out.size(2))
    if encoder_padding_mask is not None:
        encoder_padding_mask = encoder_padding_mask.unsqueeze(1).repeat(1,beam_size,1).view(bsz * beam_size, -1 )
    if decoder_padding_mask is not None:
        decoder_padding_mask = decoder_padding_mask.unsqueeze(1).repeat(1,beam_size,1).view(bsz * beam_size, -1 )
    if causal_mask is not None:
        causal_mask = causal_mask

    return encoder_out, encoder_padding_mask, decoder_padding_mask, causal_mask    
        
def predict_length_beam(gold_target_len, predicted_lengths, length_beam_size):
    if gold_target_len is not None:
        beam_starts = gold_target_len - (length_beam_size - 1) // 2
        beam_ends = gold_target_len + length_beam_size // 2 + 1
        beam = torch.stack([torch.arange(beam_starts[batch], beam_ends[batch], device=beam_starts.device) for batch in range(gold_target_len.size(0))], dim=0)
    else:
        beam = predicted_lengths.topk(length_beam_size, dim=1)[1]
    beam[beam < 2] = 2
    return beam

def outputs_to_tgt_tokens( outputs, img_seq_len, device ):

    outputs = outputs[:,img_seq_len:,:]
    outputs = model.ln_outputs( outputs )
    logits = model.linear( outputs )
    outputs = F.softmax( logits, dim = 2 )
    tgt_tokens = torch.argmax( logits, dim = 2 )
    token_probs = torch.max( outputs, dim = 2 )[1]
    
    return tgt_tokens, token_probs

def build_bert_in_and_masks( memory, masked_captions):

    emb_caption = model.emb( masked_captions ) * math.sqrt(model.dim_embedding)
    emb_caption += model.pos_emb( emb_caption )
    bert_in = torch.cat( [memory, emb_caption], dim = 1 )
          
    bert_in_padding_masks = torch.ne( masked_captions, model.pad_token_id ).float()
    bert_in_padding_masks = torch.cat( [torch.ones( memory.shape[:2], device = device ), bert_in_padding_masks], dim = 1 )   

    return bert_in, bert_in_padding_masks

# 推論モジュール
@torch.no_grad()
def inference3(
            images, length_beam_size, is_inference = True
            ):
    ''' ネットワーク計算(forward処理)の関数
    input_sequence: 各発話の入力系列 [B x Tin x D]
    input_lengths:  各発話の系列長(フレーム数) [B]
        []の中はテンソルのサイズ
        B:    ミニバッチ内の発話数(ミニバッチサイズ)
        Tin:  入力テンソルの系列長(ゼロ埋め部分含む)
        D:    入力次元数(dim_in)
        Tout: 正解ラベル系列の系列長(ゼロ埋め部分含む)
    '''

    memory = model.clip_model( images )
    memory = model.dense_connector( memory )
    memory = model.dropout( memory )
    memory = model.ln_memory( memory )
    memory = model.conv1( memory.transpose(1,2) ).transpose(1,2)
    img_seq_len = memory.size(1)

    predicted_lengths = model.lengths_predictor( memory )
    beam = predict_length_beam( None, predicted_lengths, length_beam_size)
    max_len = beam.max().item()
    bsz = memory.size(0)
    
    length_mask = torch.triu( memory.new( max_len,max_len).fill_(1).long(),1 )
    length_mask = torch.stack([length_mask[beam[batch] - 1 ] for batch in range(bsz)], dim = 0)
    tgt_tokens = memory.new( bsz, length_beam_size, max_len ).fill_(model.mask_token_id).long()
    tgt_tokens = ( 1 - length_mask ) * tgt_tokens + length_mask * model.pad_token_id
    tgt_tokens = tgt_tokens.view( bsz * length_beam_size, max_len )
    
    def select_worst(token_probs, num_mask):
        bsz, seq_len = token_probs.size()
        masks = [token_probs[batch, :].topk(max(1, num_mask[batch]), largest=False, sorted=False)[1] for batch in range(bsz)]
        masks = [torch.cat([mask, mask.new(seq_len - mask.size(0)).fill_(mask[0])], dim=0) for mask in masks]
        return torch.stack(masks, dim=0)             

    def assign_single_value_long(x, i, y):
        b, l = x.size()
        i = i + torch.arange(0, b*l, l, device=i.device).unsqueeze(1)
        x.view(-1)[i.view(-1)] = y
        return x

    def assign_single_value_byte(x, i, y):
        x.view(-1)[i.view(-1).nonzero()] = y
        return x
    
    def assign_multi_value_long(x, i, y):
        b, l = x.size()
        i = i + torch.arange(0, b*l, l, device=i.device).unsqueeze(1)
        x.view(-1)[i.view(-1)] = y.view(-1)[i.view(-1)]
        return x
    
    encoder_out = memory
    encoder_out, _, _, _ = duplicate_encoder_out( encoder_out, None, None, None, bsz, length_beam_size)        
    
    bsz, seq_len = tgt_tokens.size()
    pad_mask = tgt_tokens.eq(model.pad_token_id)
    seq_lens = seq_len - pad_mask.sum(dim=1)
    
    iter_max = 10

    masked_captions = tgt_tokens
    memory = encoder_out
    bert_in, bert_in_padding_masks =  build_bert_in_and_masks( memory, masked_captions)
    outputs = model.bert( inputs_embeds = bert_in, attention_mask = bert_in_padding_masks ).last_hidden_state
    tgt_tokens, token_probs = outputs_to_tgt_tokens( outputs, img_seq_len, encoder_out.device )
    
    tgt_tokens = assign_single_value_byte(tgt_tokens, pad_mask, model.pad_token_id )
    token_probs = assign_single_value_byte(token_probs, pad_mask, 1.0)
    
    for counter in range( 1, iter_max ):
        num_mask = ( seq_lens.float() * ( 1.0 - ( counter / iter_max))).long()

        assign_single_value_byte(token_probs, pad_mask, 1.0)
        mask_ind = select_worst(token_probs, num_mask)

        tgt_tokens = assign_single_value_long(tgt_tokens, mask_ind, model.mask_token_id)
        tgt_tokens = assign_single_value_byte(tgt_tokens, pad_mask, model.pad_token_id)    

        masked_captions = tgt_tokens
        bert_in, bert_in_padding_masks =  build_bert_in_and_masks( memory, masked_captions)
        outputs = model.bert( inputs_embeds = bert_in, attention_mask = bert_in_padding_masks ).last_hidden_state
        new_tgt_tokens, new_token_probs = outputs_to_tgt_tokens( outputs, img_seq_len, encoder_out.device )

        token_probs = assign_multi_value_long(token_probs, mask_ind, new_token_probs)
        token_probs = assign_single_value_byte(token_probs, pad_mask, 1.0)
            
        tgt_tokens = assign_multi_value_long(tgt_tokens, mask_ind, new_tgt_tokens)
        tgt_tokens = assign_single_value_byte(tgt_tokens, pad_mask, model.pad_token_id)
        
    lprobs = token_probs.log().sum(-1)

    return tgt_tokens, lprobs, max_len, length_mask


### テスト

In [ ]:
#test_pr_coef = len( test_loader ) // 20
test_pr_coef = 1

fn = bleu_score.SmoothingFunction().method7

transforms_inv = v2.Compose([
    v2.Normalize((-0.48145466/0.26862954, -0.4578275/0.26130258, -0.40821073/0.27577711), (1/0.26862954,1/0.26130258,1/0.27577711)),
    v2.ToPILImage()
])

# 検証
with tqdm(test_loader) as pbar:
    pbar.set_description(f'[テスト]')

    # 評価モード
    model.eval()

    test_errors = deque()
    test_bleus = deque()
    n_batch = 0
    length_max = 100
    for k, (imgs, captions, caption_lengths) in enumerate( pbar ):
        if k > 20:
            break
        # ミニバッチを設定
        imgs = imgs.to(device)
        captions = captions.to(device)
        #caption_lengths = torch.tensor( caption_lengths ).to(config.device)
        
        with torch.no_grad():
            #infe = 1
            infe = 2
            #infe = 3
            if infe == 1:
                length_beam_size = 1
                prop_logits = inference(imgs, tokenizer )
                hypo_ids = torch.argmax( prop_logits, dim = 2 )
            elif infe == 2:
                length_beam_size = 3
                captions2, mean_prob = inference2(imgs, length_beam_size, tokenizer )
                bsz = imgs.size(0)
                captions2 = captions2.view( bsz, length_beam_size, -1 )
                mean_prob = mean_prob.view( bsz, length_beam_size )
                best_lengths = mean_prob.max(-1)[1]
                captions2 = torch.stack([captions2[b, l, :] for b, l, in enumerate(best_lengths)], dim = 0 )
                hypo_ids = captions2
            elif infe == 3:
                length_beam_size = 3
                preds, lprobs, max_len, length_mask = inference3( imgs, length_beam_size )
                hypotheses = preds
                bsz = imgs.size(0)
                hypotheses = hypotheses.view(bsz, length_beam_size, max_len)
                lprobs = lprobs.view(bsz, length_beam_size)
                tgt_lengths = (1 - length_mask).sum(-1)
                avg_log_prob = lprobs / tgt_lengths.float()
                best_lengths = avg_log_prob.max(-1)[1]
                hypotheses = torch.stack([hypotheses[b, l, :] for b, l in enumerate(best_lengths)], dim=0)
                pred = torch.squeeze( hypotheses, dim = 0 )
                hypo_ids = [pred]
        
        n = 0
        hypo_sentence = []
        ref_sentence = []
        ref_imgs = []
        total_error = 0
        total_token_length = 0
        total_bleu = 0
        for (hypo_id, caption, img ) in zip( hypo_ids, captions, imgs ):
            hypo = model.my_decode( hypo_id.tolist(), tokenizer )
            hypo_tokens = tokenizer.tokenize( hypo )
            reference = model.my_decode( caption.tolist(), tokenizer )
            ref_tokens = tokenizer.tokenize( reference )

            
            # 認識誤りを計算
            (error, substitute, delete, insert, ref_length) = levenshtein.calculate_error(hypo_tokens,ref_tokens)
            
            # 誤り文字数を累積する
            total_error += error
            # 文字の総数を累積する
            total_token_length += ref_length

            bleu = bleu_score.sentence_bleu( [reference], hypo, smoothing_function=fn  )
        
            total_bleu += bleu
            
            #if n < 1 and n_batch % test_pr_coef == 0:
            #print( "hypo:", hypo )
            #print( "reference:", reference )
            hypo_sentence.append( hypo )
            ref_sentence.append( reference )
            ref_imgs.append( img )
                   
            n += 1
                
        n_batch += 1
        avg_error = total_error / total_token_length * 100
        print( "this pic. WER :", avg_error )
        avg_bleu = total_bleu / n * 100
        print( "this pic. BLEU:", avg_bleu )
                
        test_errors.append(avg_error)
        test_bleus.append(avg_bleu)
                
        #if len(test_errors) > config.moving_avg:
        if len(test_errors) > 100:
            test_errors.popleft()
            test_bleus.popleft()
        pbar.set_postfix({
            #'loss': torch.Tensor(test_losses).mean().item(),
            'WER': torch.Tensor(test_errors).mean().item(),
            'BLEU': torch.Tensor(test_bleus).mean().item()
        })                
                
                    
        for ( hypo_se, ref_se, img ) in zip( hypo_sentence, ref_sentence, ref_imgs ):
            print(f'test number = {k} average, WER = {torch.Tensor(test_errors).mean().item()}, BLEU = {torch.Tensor(test_bleus).mean().item()}')
            print( "refe:", ref_se )
            print( "hypo:", hypo_se )
            inv_img = transforms_inv( img )
            plt.imshow( inv_img )
            plt.axis('off')
            plt.show()


# 表示
test_error = np.mean( test_errors )
test_bleu = np.mean( test_bleus )
print(f'test {k} average WER : {test_error}')
print(f'test {k} average BLEU: {test_bleu}')

In [ ]:
reference="A selection of scissors, including pinking shears, under a glass shelf."
ref = reference.split( " " )
hypo="A bunch of scissors that are on a table.OTHER.R."
hyp = hypo.split( " " )

(error, substitute, delete, insert, ref_length) = levenshtein.calculate_error(hyp,ref)

print( error, substitute, delete, insert, ref_length )
print( error / ref_length * 100 )